## §0 Setup and Methodology

In [ ]:
import sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})
warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'notebooks'))

from _shared import (
    ann_sharpe, ann_return, ann_vol, max_drawdown,
    apply_vmp_overlay, TRADING_DAYS,
    FAMILY_COLORS, DISPLAY,
)
from aiam.data.panel import Panel
from aiam.data.split import TRAIN_END, TEST_START
from aiam.features.technical import (
    momentum, volatility, rsi, atr, bollinger, gap, volume_signal, forward_returns
)
from aiam.features.asset_class import asset_class_one_hot
from aiam.ml.workflow import chronological_splits, leakage_check_forward_returns
from aiam.strategy.ml_strategies import LassoSignalStrategy, RFSignalStrategy, XGBSignalStrategy
from aiam.strategy.signal_tilt import SignalTilt, momentum_signal_fn
from aiam.estimators.covariance import ledoit_wolf_cov

print(f'ROOT: {ROOT}')
print(f'Train end: {TRAIN_END}  |  Test start: {TEST_START}')

## Paper 2 — Machine Learning Strategies

Paper 1 established the deterministic bar: 62 strategies across six classical families evaluated walk-forward on 2003–2026. The full-sample leader, excluding the degenerate VMP(GMV(sample)) SHY-concentration artifact, was *VMP(MDP(LW))* at Sharpe 1.372; the best-in-study including the regime-switching extension was *VMP(SWITCH(v2a))* at 1.660. The honest OOS leader on the 2023–2026 test window alone was *VMP(MDP(LW))* at ~2.43. Paper 2 asks a harder question — do *learned* approaches add genuine skill beyond that bar?

The epistemic shift is real. Classical strategies carry no training constraint; they can implicitly use the full sample's covariance structure and regime history. ML models must commit their parameters before the test period begins, facing overfitting risk, hyperparameter sensitivity, and regime shift as live concerns. The fair question is therefore not "what is ML's full-sample Sharpe?" but "does ML beat the best deterministic family on the **same** 2023–2026 window?" Notebook 00's *Evaluation discipline* section is the canonical framing.

This chapter covers three model families (Lasso, RF, XGBoost), two portfolio-construction wrappers (SignalTilt tilt overlay and MSR with ML-predicted μ̂), a multi-model ensemble, and walk-forward re-estimation. The apples-to-apples OOS-vs-OOS verdict is in **§14b** — that is where the real comparison lives.

**Methodology note.** ML models are trained once on the full training window (2003–2019, ~17 years) and evaluated out-of-sample on 2023–2026 (~3.3 years). Classical paper baselines use full-sample Sharpes (2003–2026). This is an intentional apples-to-pears comparison: the ML evaluation is methodologically more rigorous (true OOS), while classical strategies are not penalized for in-sample fitting. ML results live in this notebook only — the master table (Session 1) is unchanged.

**What we test.** *Approach B (core)*: SignalTilt-wrapped ML cross-sectional scores vs. the 1.5B momentum baseline. *Approach A (extension)*: MSR with ML-predicted μ̂ inputs.

In [ ]:
# Load returns and prices
returns = pd.read_parquet(ROOT / 'data/cache/returns_29_2003_2026.parquet')
returns.index = pd.to_datetime(returns.index)
returns.index.name = 'Date'
returns.columns.name = 'Asset'

prices = pd.read_parquet(ROOT / 'data/cache/prices_29.parquet')
prices.index = pd.to_datetime(prices.index)
prices.index.name = 'Date'
prices.columns.name = 'Asset'

# Published strategy returns for paper baselines
base_pub = pd.read_parquet(ROOT / 'data/published/strategy_returns_base.parquet')
base_pub.index = pd.to_datetime(base_pub.index)
vmp_pub = pd.read_parquet(ROOT / 'data/published/strategy_returns_vmp.parquet')
vmp_pub.index = pd.to_datetime(vmp_pub.index)

# SWITCH(v2a)
sw_oos = pd.read_parquet(ROOT / 'data/cache/portfolio_returns/switch_v2a_oos_29assets.parquet')
sw_oos.index = pd.to_datetime(sw_oos.index)
switch_v2a = sw_oos['SWITCH_v2a_train_only'].rename('SWITCH(v2a)')

print(f'Returns: {returns.shape}, {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'Prices: {prices.shape}')
print(f'Published base: {base_pub.shape}')

## §1 Feature Engineering for ML
What signals feed the models, and how is look-ahead leakage prevented? Ten numeric features are constructed from prices and returns — momentum at three horizons (21d, 63d, 252d), realised volatility (60d), RSI, ATR, Bollinger z-score, gap, volume signal, and asset-class one-hot dummies — using only data available strictly before each observation date. The forward-return target is computed with a one-step lag; a spot-check leakage audit confirms no future information bleeds into training.

In [ ]:
# Load OHLCV, unstack to wide format
ohlcv_raw = pd.read_parquet(ROOT / 'data/cache/prices_29_ohlcv_2003_2026.parquet')
ohlcv_raw.index = pd.MultiIndex.from_tuples(
    [(pd.Timestamp(d), t) for d, t in ohlcv_raw.index],
    names=['Date', 'Asset']
)

prices_close  = ohlcv_raw['close'].unstack('Asset')
prices_open   = ohlcv_raw['open'].unstack('Asset')
prices_high   = ohlcv_raw['high'].unstack('Asset')
prices_low    = ohlcv_raw['low'].unstack('Asset')
prices_volume = ohlcv_raw['volume'].unstack('Asset')

# Use returns from cache (consistent with pipeline)
rets = returns.copy()

ohlc_dict = {'open': prices_open, 'high': prices_high, 'low': prices_low,
              'close': prices_close, 'volume': prices_volume}

print(f'prices_close: {prices_close.shape}, volume: {prices_volume.shape}')

In [ ]:
# Compute 10 numeric features
feat_mom_21   = momentum(rets, 21)
feat_mom_63   = momentum(rets, 63)
feat_mom_252  = momentum(rets, 252)
feat_vol_60   = volatility(rets, 60)
feat_vol_252  = volatility(rets, 252)
feat_rsi_14   = rsi(prices_close, 14)
feat_atr_raw  = atr(ohlc_dict, 14)
feat_atr_ratio = feat_atr_raw / prices_close  # scale-invariant
bb = bollinger(prices_close, window=20)
feat_bb_pct   = bb['pct']
feat_gap      = gap(ohlc_dict)
feat_vol_sig  = volume_signal(prices_volume, lookback=21)

# Stack to long form
numeric_frames = {
    'mom_21':       feat_mom_21,
    'mom_63':       feat_mom_63,
    'mom_252':      feat_mom_252,
    'vol_60':       feat_vol_60,
    'vol_252':      feat_vol_252,
    'rsi_14':       feat_rsi_14,
    'atr_14_ratio': feat_atr_ratio,
    'bb_pct':       feat_bb_pct,
    'gap':          feat_gap,
    'vol_signal_21': feat_vol_sig,
}
panel_numeric = pd.concat({k: v.stack() for k, v in numeric_frames.items()}, axis=1)
panel_numeric.index.names = ['Date', 'Asset']

# Join 7 asset-class one-hots
assets = rets.columns.tolist()
oh = asset_class_one_hot(assets)
feature_panel = panel_numeric.join(oh, on='Asset')

# Drop rows where warmup-heavy features are NaN
warmup_cols = ['mom_252', 'vol_252']
feature_panel = feature_panel.dropna(subset=warmup_cols)

NUMERIC_COLS = list(numeric_frames.keys())
AC_COLS = list(oh.columns)
FEATURE_COLS = NUMERIC_COLS + AC_COLS

print(f'Feature panel: {feature_panel.shape}')
print(f'Feature cols ({len(FEATURE_COLS)}): {FEATURE_COLS}')
feature_panel.head(3)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
corr = feature_panel[FEATURE_COLS].dropna().corr().values
labels = FEATURE_COLS
cmap = plt.cm.RdBu_r
im = ax.pcolormesh(corr, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(labels)) + 0.5)
ax.set_yticks(np.arange(len(labels)) + 0.5)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
for i in range(len(labels)):
    for j in range(len(labels)):
        v = corr[i, j]
        ax.text(j + 0.5, i + 0.5, f'{v:.2f}', ha='center', va='center',
                fontsize=4.5, color='white' if abs(v) > 0.6 else 'black')
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
ax.set_title(f'Feature correlation matrix ({len(labels)} features)', fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
fig_dir = ROOT / 'docs/figures/session2'
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / 'feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session2/feature_correlation.png')


## §2 Target Construction and Leakage Verification
ML models need a supervised target. We define it as the 21-day forward return — the outcome each model is trained to predict. The leakage check confirms that every target observation sits strictly *after* its feature date: using future prices to predict future prices is the most common silent bug in financial ML, and we verify it explicitly before training.

In [ ]:
HORIZON = 21
fwd_ret_wide = forward_returns(rets, HORIZON)
target_panel = fwd_ret_wide.stack()
target_panel.index.names = ['Date', 'Asset']
target_panel.name = 'target_21d'

# Align feature and target panels on common index
common_idx = feature_panel.index.intersection(target_panel.dropna().index)
X_full = feature_panel.loc[common_idx]
y_full = target_panel.loc[common_idx]

print(f'Target panel: {target_panel.shape}, X_full: {X_full.shape}, y_full: {y_full.shape}')

In [ ]:
# Leakage verification — spot-check 3 (asset, date) pairs
check_pairs = [
    ('AAPL.US', pd.Timestamp('2010-06-15')),
    ('GLD.US',  pd.Timestamp('2015-03-20')),
    ('TLT.US',  pd.Timestamp('2019-11-05')),
]
print('Leakage check:')
all_pass = True
for asset, date in check_pairs:
    ok = leakage_check_forward_returns(rets, fwd_ret_wide, HORIZON, asset, date)
    print(f'  {asset} @ {date.date()}: {"PASS" if ok else "FAIL"}')
    all_pass = all_pass and ok
assert all_pass, 'Leakage check FAILED'
print('All leakage checks PASSED ✓')

## §3 Train/Validation/Test Splits
Data is split chronologically: ~80% for training (2003–~2019), ~15% for validation (hyperparameter sensitivity), and the held-out test period (2023–2026) for final OOS evaluation. No shuffling, no cross-validation across time — financial data has strong serial autocorrelation and regime persistence that render i.i.d. assumptions invalid.

In [ ]:
panel_dates = X_full.index.get_level_values('Date').unique().sort_values()
train_dates, val_dates, test_dates = chronological_splits(
    panel_dates, train_end=TRAIN_END, test_start=TEST_START, validation_share=0.15
)

print(f'Train:      {train_dates[0].date()} → {train_dates[-1].date()}  ({len(train_dates)} days, {len(train_dates)*29:,} obs)')
print(f'Validation: {val_dates[0].date()} → {val_dates[-1].date()}  ({len(val_dates)} days, {len(val_dates)*29:,} obs)')
print(f'Test:       {test_dates[0].date()} → {test_dates[-1].date()}  ({len(test_dates)} days, {len(test_dates)*29:,} obs)')

In [ ]:
STRATEGY_FAMILY = {
    'EW': 'Classical', 'MSR(LW)': 'Classical', 'SWITCH(v2a)': 'Classical', 'VMP(MDP(LW))': 'Classical',
    'SignalTilt(mom_252)': 'Classical',
    'SignalTilt(Lasso)': 'ML single-fit', 'SignalTilt(RF)': 'ML single-fit', 'SignalTilt(XGB)': 'ML single-fit',
    'MSR(Lasso_μ̂)': 'ML single-fit', 'MSR(RF_μ̂)': 'ML single-fit', 'MSR(XGB_μ̂)': 'ML single-fit',
    'VMP(SignalTilt(Lasso))': 'ML + VMP', 'VMP(SignalTilt(RF))': 'ML + VMP', 'VMP(SignalTilt(XGB))': 'ML + VMP',
    'VMP(MSR(Lasso_μ̂))': 'ML + VMP', 'VMP(MSR(RF_μ̂))': 'ML + VMP', 'VMP(MSR(XGB_μ̂))': 'ML + VMP',
    'SignalTilt(Ensemble)': 'ML ensemble', 'MSR(Ensemble_μ̂)': 'ML ensemble',
}
ML_FAMILY_COLORS = {
    'Classical': '#1f4e79', 'ML single-fit': '#d62728',
    'ML + VMP': '#ff7f0e', 'ML ensemble': '#2ca02c',
}
print('Strategy families:', {v: sum(1 for x in STRATEGY_FAMILY.values() if x == v) for v in ML_FAMILY_COLORS})


## §4 Lasso Signal Strategy
Three model families — Lasso (linear with L1 regularisation), Random Forest (bagged trees), and XGBoost (gradient-boosted trees) — are each trained *once* on the 2003–2019 window and never retrained. Single-fit methodology keeps the OOS comparison clean and mirrors the discipline of Paper 1's classical strategies. The cost is regime blindness: a model that learned pre-2020 relationships may not generalise cleanly to a post-rate-shock environment; §17 tests whether walk-forward retraining helps.

In [ ]:
t0 = time.time()
lasso_strat = LassoSignalStrategy(
    feature_panel=X_full,
    target_panel=y_full,
    feature_cols=FEATURE_COLS,
    train_end=TRAIN_END,
    validation_share=0.15,
    alpha=1e-4,
    tilt_strength=0.5,
)
lasso_time = time.time() - t0
print(f'Lasso fit time: {lasso_time:.1f}s')
print(f'Predictions shape: {lasso_strat.predictions.shape}')
print(f'Non-zero Lasso coefficients: {(lasso_strat.model.coef_ != 0).sum()} / {len(lasso_strat.model.coef_)}')
lasso_strat.predictions.head()

## §5 Random Forest Signal Strategy

In [ ]:
t0 = time.time()
rf_strat = RFSignalStrategy(
    feature_panel=X_full,
    target_panel=y_full,
    feature_cols=FEATURE_COLS,
    train_end=TRAIN_END,
    validation_share=0.15,
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=50,
    tilt_strength=0.5,
)
rf_time = time.time() - t0
print(f'RF fit time: {rf_time:.1f}s')
print(f'Predictions shape: {rf_strat.predictions.shape}')
rf_strat.predictions.head()

## §6 XGBoost Signal Strategy

In [ ]:
t0 = time.time()
xgb_strat = XGBSignalStrategy(
    feature_panel=X_full,
    target_panel=y_full,
    feature_cols=FEATURE_COLS,
    train_end=TRAIN_END,
    validation_share=0.15,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    tilt_strength=0.5,
)
xgb_time = time.time() - t0
print(f'XGBoost fit time: {xgb_time:.1f}s')
print(f'Predictions shape: {xgb_strat.predictions.shape}')
xgb_strat.predictions.head()

In [ ]:
from scipy.stats import spearmanr

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
rng = np.random.default_rng(42)
for ax, (name, strat) in zip(axes, [('Lasso', lasso_strat), ('RF', rf_strat), ('XGB', xgb_strat)]):
    test_mask_p = strat.predictions.index.get_level_values('Date') >= pd.Timestamp(TEST_START)
    pred_test = strat.predictions[test_mask_p]
    real_test = y_full.reindex(pred_test.index).dropna()
    pred_aligned = pred_test.reindex(real_test.index)
    rank_ic = spearmanr(pred_aligned.values, real_test.values)[0]
    if len(real_test) > 5000:
        idx = rng.choice(len(real_test), 5000, replace=False)
        p, r = pred_aligned.iloc[idx], real_test.iloc[idx]
    else:
        p, r = pred_aligned, real_test
    color = ML_FAMILY_COLORS['ML single-fit']
    ax.scatter(p.values, r.values, alpha=0.15, s=3, color=color, rasterized=True)
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.axvline(0, color='gray', lw=0.5, ls='--')
    ax.set_xlabel('Predicted z-score', fontsize=9)
    ax.set_title(f'{name}  Rank-IC={rank_ic:.3f}', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Realized 21d return', fontsize=9)
plt.suptitle('Predicted vs Realized (test period 2023–2026, 5000 samples each)', fontsize=11)
plt.tight_layout()
fig_dir = ROOT / 'docs/figures/session2'
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / 'predicted_vs_realized.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session2/predicted_vs_realized.png')


## §7 Permutation Importance (RF)
Which features actually drive the RF's predictions? Permutation importance randomly shuffles each feature column and measures the drop in validation-set IC — the larger the drop, the more the model relied on that feature. This gives a model-agnostic attribution that avoids the cardinality bias of split-based importance, and it connects back to the Session 1.5B IC findings for ground-truth validation.

In [ ]:
from sklearn.inspection import permutation_importance as _pi

t0 = time.time()
_pi_res = _pi(rf_strat.model, rf_strat._X_val, rf_strat._y_val, n_repeats=10, random_state=42)
rf_importance = pd.Series(_pi_res.importances_mean, index=rf_strat._feature_cols)
imp_std = pd.Series(_pi_res.importances_std, index=rf_strat._feature_cols)
print(f'Permutation importance (n_repeats=10): {time.time()-t0:.1f}s')

top15_idx = rf_importance.sort_values().tail(15).index
top15_mean = rf_importance.reindex(top15_idx)
top15_std = imp_std.reindex(top15_idx)
colors = [ML_FAMILY_COLORS['ML single-fit'] if v >= 0 else '#aaaaaa' for v in top15_mean.values]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(range(len(top15_mean)), top15_mean.values, xerr=top15_std.values,
        color=colors, ecolor='#444444', capsize=3, alpha=0.85)
ax.set_yticks(range(len(top15_mean)))
ax.set_yticklabels(top15_mean.index, fontsize=9)
ax.axvline(0, color='black', linewidth=0.7)
ax.set_xlabel('Mean Importance ± σ (validation set, n_repeats=10)')
ax.set_title('Random Forest — Permutation Importance with Error Bars')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig_dir = ROOT / 'docs/figures/session2'
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / 'rf_permutation_importance.png', dpi=150, bbox_inches='tight')
fig.savefig(ROOT / 'docs/figures/rf_permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session2/rf_permutation_importance.png')
print('\nTop 5 features by RF permutation importance:')
print(rf_importance.sort_values(ascending=False).head(5).to_string())


Permutation importance on the validation set reveals which features the RF relies on most. Momentum (252-day) and volatility features typically dominate: consistent with the 1.5B finding that both momentum and vol carry cross-sectional predictive power on this 29-asset universe. Asset-class dummies contribute moderate importance, confirming that within-class structure (e.g. fixed income vs. equities) is informative. Short-horizon momentum (mom_21) tends to rank low, matching its near-zero IC found in Session 1.5B.

## §8 Approach B — SignalTilt-Wrapped ML Comparison
The simplest way to exploit an ML signal in a portfolio is the *tilt* overlay: rank assets by the model's cross-sectional score and tilt away from equal-weight proportional to that rank. This preserves broad diversification while injecting signal, and it is less sensitive to ML prediction errors than a full mean-variance optimiser. The 252-day momentum baseline (Sharpe ~1.156 OOS, from Session 1.5B) is the first bar Approach B must clear.

In [ ]:
# Build Panel for walk-forward evaluation
panel = Panel({'prices': prices, 'returns': returns})

# Test dates for evaluation: next trading day after TEST_START
eval_dates = returns.index[returns.index >= TEST_START]

# SignalTilt(mom_252) — 1.5B baseline
tilt_mom = SignalTilt(momentum_signal_fn, tilt_strength=0.5, signal_lookback=252)

def walk_forward_returns(strat, dates, returns_wide, panel_obj):
    """Walk-forward strategy returns: weights on date t applied to t+1 return."""
    ret_series = {}
    valid_dates = dates[:-1]  # last date has no next-day return
    for d in valid_dates:
        w = strat.predict_weights(panel_obj, d)
        next_d_mask = returns_wide.index > d
        if not next_d_mask.any():
            continue
        next_d = returns_wide.index[next_d_mask][0]
        r_next = returns_wide.loc[next_d]
        ret_series[next_d] = (w.reindex(r_next.index).fillna(0) * r_next).sum()
    return pd.Series(ret_series)

print('Running walk-forward evaluation (Approach B)...')
t0 = time.time()
ret_ew = walk_forward_returns(
    type('EW', (), {'predict_weights': lambda self, p, d: pd.Series(1/len(p.universe_at(d)), index=p.universe_at(d))})(),
    eval_dates, returns, panel
)
print(f'  EW done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_tilt_mom = walk_forward_returns(tilt_mom, eval_dates, returns, panel)
print(f'  SignalTilt(mom_252) done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_lasso = walk_forward_returns(lasso_strat, eval_dates, returns, panel)
print(f'  Lasso done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_rf = walk_forward_returns(rf_strat, eval_dates, returns, panel)
print(f'  RF done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_xgb = walk_forward_returns(xgb_strat, eval_dates, returns, panel)
print(f'  XGB done ({time.time()-t0:.0f}s)')

# Align on common test dates
test_mask = ret_lasso.index >= TEST_START
test_idx = ret_lasso.index[test_mask]

def metrics(r, name):
    r = r.reindex(test_idx).dropna()
    return {
        'Strategy': name,
        'Ann Ret': ann_return(r),
        'Ann Vol': ann_vol(r),
        'Sharpe': ann_sharpe(r),
        'Max DD': max_drawdown(r),
    }

approach_b = pd.DataFrame([
    metrics(ret_ew,       'EW'),
    metrics(ret_tilt_mom, 'SignalTilt(mom_252)'),
    metrics(ret_lasso,    'SignalTilt(Lasso)'),
    metrics(ret_rf,       'SignalTilt(RF)'),
    metrics(ret_xgb,      'SignalTilt(XGB)'),
]).set_index('Strategy')

print('\nApproach B — Test period metrics:')
print(approach_b.map(lambda x: f'{x:.3f}').to_string())

In [ ]:
from aiam.features.asset_class import ASSET_CLASS_MAP as _ACM

_CLASSES = ['us_single_stock', 'us_sector_etf', 'broad_equity_etf', 'intl_equity_etf',
            'fixed_income_etf', 'commodity_etf', 'fx_spot']
_CLASS_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#7f7f7f']

sample_dates = eval_dates[::21]
wt_records = []
for d in sample_dates:
    w = xgb_strat.predict_weights(panel, d)
    row = {cls: 0.0 for cls in _CLASSES}
    for asset, wt in w.items():
        cls = _ACM.get(asset, 'us_single_stock')
        row[cls] = row.get(cls, 0.0) + float(wt)
    row['Date'] = d
    wt_records.append(row)
wt_df = pd.DataFrame(wt_records).set_index('Date')[_CLASSES]

fig, ax = plt.subplots(figsize=(11, 4))
ax.stackplot(wt_df.index, wt_df.values.T, labels=_CLASSES, colors=_CLASS_COLORS, alpha=0.85)
ax.set_ylabel('Portfolio weight', fontsize=10)
ax.set_title('SignalTilt(XGB) — Asset class allocation (monthly sample, test period 2023–2026)', fontsize=11)
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.set_ylim(0, 1)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig_dir = ROOT / 'docs/figures/session2'
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / 'weight_evolution_xgb.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session2/weight_evolution_xgb.png')


## §9 Approach A — MSR with ML-Predicted Returns
Approach A feeds the ML predicted μ̂ directly into the MSR optimiser — treating the ML output as a genuine expected-return vector. This is the highest-conviction use of the signal but also the most fragile: MSR is well-known to amplify estimation error (Michaud 1989), turning small μ̂ miscalibrations into extreme weight concentrations. A long-only constraint and Ledoit–Wolf covariance shrinkage damp but do not eliminate this instability.

MSR is known to be unstable to mean inputs (Michaud 1989). The long-only clip + renormalize is a defensive measure but does not fix the fundamental sensitivity. Expect higher volatility than SignalTilt wrapping. Risk-free rate is set to zero for simplicity (documented assumption).

In [ ]:
from aiam.ml.workflow import cross_sectional_score

def msr_ml_returns(strat, eval_dates, returns_wide, lookback_cov=504):
    """MSR with ML-predicted mu_hat. Long-only, renormalized."""
    ret_series = {}
    valid_dates = eval_dates[:-1]
    for d in valid_dates:
        mu_hat = cross_sectional_score(strat.predictions, d)
        if mu_hat.empty:
            continue
        # Trailing covariance
        r_window = returns_wide.loc[returns_wide.index <= d].tail(lookback_cov)
        r_clean = r_window[mu_hat.index].dropna(how='any')
        if len(r_clean) < 60:
            continue
        sigma = ledoit_wolf_cov(r_clean)
        try:
            sigma_inv = np.linalg.inv(sigma)
        except np.linalg.LinAlgError:
            continue
        mu = mu_hat.values
        w_unc = sigma_inv @ mu
        w_lo = np.clip(w_unc, 0, None)
        total = w_lo.sum()
        if total < 1e-12:
            w_final = pd.Series(1.0 / len(mu_hat), index=mu_hat.index)
        else:
            w_final = pd.Series(w_lo / total, index=mu_hat.index)
        # Next-day return
        next_d_mask = returns_wide.index > d
        if not next_d_mask.any():
            continue
        next_d = returns_wide.index[next_d_mask][0]
        r_next = returns_wide.loc[next_d]
        ret_series[next_d] = (w_final.reindex(r_next.index).fillna(0) * r_next).sum()
    return pd.Series(ret_series)

print('Running Approach A — MSR with ML μ̂...')
t0 = time.time()
ret_msr_lasso = msr_ml_returns(lasso_strat, eval_dates, returns)
print(f'  MSR(Lasso) done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_msr_rf = msr_ml_returns(rf_strat, eval_dates, returns)
print(f'  MSR(RF) done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_msr_xgb = msr_ml_returns(xgb_strat, eval_dates, returns)
print(f'  MSR(XGB) done ({time.time()-t0:.0f}s)')

approach_a = pd.DataFrame([
    metrics(ret_msr_lasso, 'MSR(Lasso_μ̂)'),
    metrics(ret_msr_rf,    'MSR(RF_μ̂)'),
    metrics(ret_msr_xgb,   'MSR(XGB_μ̂)'),
]).set_index('Strategy')

print('\nApproach A — Test period metrics:')
print(approach_a.map(lambda x: f'{x:.3f}').to_string())

## §10 Headline Comparison Table
The headline table ranks all strategies by OOS Sharpe on 2023–2026. One note on framing: the classical benchmarks here use *published* full-sample returns — an intentional apples-to-pears comparison that does not penalise classical strategies for in-sample fitting. **§14b immediately below** corrects this by placing ML and deterministic strategies on the same OOS window — that is the honest verdict.

In [ ]:
def turnover(strat, dates, panel_obj, lookback=1):
    """Mean absolute daily weight change (proxy for turnover)."""
    prev_w = None
    to_list = []
    for d in dates[:-1]:
        w = strat.predict_weights(panel_obj, d)
        if prev_w is not None:
            to_list.append((w.reindex(prev_w.index).fillna(0) - prev_w).abs().sum())
        prev_w = w
    return np.mean(to_list) if to_list else np.nan

# Paper baselines on test period
test_mask_pub = base_pub.index >= TEST_START
msr_lw_test = base_pub['MSR(ledoit_wolf)'][test_mask_pub]
sw_test = switch_v2a[switch_v2a.index >= TEST_START]
vmp_mdp_test = vmp_pub['VMP(MDP(ledoit_wolf))'][vmp_pub.index >= TEST_START]

def metrics_with_to(r, name, to_val=np.nan):
    r = r.reindex(test_idx).dropna()
    return {
        'Strategy': name,
        'Ann Ret': ann_return(r),
        'Ann Vol': ann_vol(r),
        'Sharpe': ann_sharpe(r),
        'Max DD': max_drawdown(r),
        'Turnover': to_val,
    }

# Approx turnover for ML strategies (sample 30 dates for speed)
sample_dates = eval_dates[:30]
to_lasso = turnover(lasso_strat, sample_dates, panel)
to_rf    = turnover(rf_strat,    sample_dates, panel)
to_xgb   = turnover(xgb_strat,  sample_dates, panel)

rows = [
    metrics_with_to(ret_ew,         'EW',                        0.0),
    metrics_with_to(ret_tilt_mom,   'SignalTilt(mom_252)',        np.nan),
    metrics_with_to(ret_lasso,      'SignalTilt(Lasso)',          to_lasso),
    metrics_with_to(ret_rf,         'SignalTilt(RF)',             to_rf),
    metrics_with_to(ret_xgb,        'SignalTilt(XGB)',            to_xgb),
    metrics_with_to(ret_msr_lasso,  'MSR(Lasso_μ̂)',              np.nan),
    metrics_with_to(ret_msr_rf,     'MSR(RF_μ̂)',                 np.nan),
    metrics_with_to(ret_msr_xgb,    'MSR(XGB_μ̂)',                np.nan),
    metrics_with_to(msr_lw_test,    'MSR(LW)',                   np.nan),
    metrics_with_to(sw_test,        'SWITCH(v2a)',               np.nan),
    metrics_with_to(vmp_mdp_test,   'VMP(MDP(LW))',              np.nan),
]

comparison = pd.DataFrame(rows).set_index('Strategy').sort_values('Sharpe', ascending=False)

# Format
fmt = comparison.copy()
fmt['Ann Ret'] = fmt['Ann Ret'].map(lambda x: f'{x:.3f}')
fmt['Ann Vol'] = fmt['Ann Vol'].map(lambda x: f'{x:.3f}')
fmt['Sharpe']  = fmt['Sharpe'].map(lambda x: f'{x:.3f}')
fmt['Max DD']  = fmt['Max DD'].map(lambda x: f'{x:.3f}')
fmt['Turnover'] = fmt['Turnover'].map(lambda x: f'{x:.4f}' if not pd.isna(x) else '—')

print('Headline Comparison Table (sorted by Sharpe, test period 2023–2026):')
print(fmt.to_string())

# Save
out_dir = ROOT / 'data/cache/portfolio_returns'
out_dir.mkdir(parents=True, exist_ok=True)
comparison_out = comparison.copy()
comparison_out.index.name = 'Strategy'
comparison_out = comparison_out.reset_index()
comparison_out.to_csv(out_dir / 'ml_strategies_comparison_table.csv', index=False)
print(f'\nSaved comparison table to {out_dir}/ml_strategies_comparison_table.csv')

## §11 Findings and Limitations
Four honest caveats bound the interpretation of everything above: (i) single-fit — models trained through 2019 are never retrained; (ii) a 3.3-year test window is narrow for robust Sharpe inference; (iii) no systematic hyperparameter search; (iv) MSR instability amplifies μ̂ errors. Read the results in light of these limits before drawing strong conclusions.

Future work — feature enrichment. The strongest next step is richer cross-sectional features (same price-derived, PIT-clean, all-asset-classes category): short-term reversal (−1-week return, a signal distinct from momentum and a classic alpha), rolling beta/correlation to SPY (each asset’s market sensitivity), risk-adjusted momentum (momentum ÷ vol), and 12-1 momentum (skip the most recent month to separate momentum from reversal) — plus cross-sectional ranking/normalization of every feature per date. All vary across assets and are defined for every asset class. Run as a baseline-vs-enriched ablation (see blueprint §5.1).

**SignalTilt(ML) vs SignalTilt(mom_252).** The three ML-wrapped strategies compete with the momentum-252 baseline on the test period. Any Sharpe advantage is modest: on a 29-asset cross-asset universe with a 3.3-year test window, noise is substantial relative to signal. The 1.5B validation showed mom_252 carries an IC of ~0.073 — a hard baseline to beat with a single-fit model trained years earlier.

**MSR(ML μ̂) vs MSR(LW).** MSR with ML inputs shows the instability Michaud (1989) predicted: even small improvements in expected-return prediction do not translate cleanly to Sharpe, because the optimizer amplifies estimation error. The long-only clip dampens but does not eliminate this sensitivity. Comparing Sharpes of MSR(Lasso_μ̂), MSR(RF_μ̂), MSR(XGB_μ̂) vs MSR(LW) illustrates why robust estimators (Black-Litterman, shrinkage) outperform raw ML inputs in production.

**Permutation importance findings.** RF relied most heavily on momentum and volatility features, matching the 1.5B cross-asset IC findings. The positive role of `vol_60` confirms the Session 1.5B result: on a cross-asset universe the cross-asset risk premium dominates the intra-equity low-vol anomaly, so higher-vol assets receive higher predicted returns. Asset-class one-hots appear with moderate importance, confirming within-class heterogeneity is informative.

**Limitations.** (i) Single-fit model: trained 2003–2019, never retrained. Financial regimes shift; a rolling-refit (walk-forward cross-validation, Session 2c) would likely close some of the gap vs. baselines. (ii) 3.3-year test window is narrow for robust inference on annual Sharpes; standard errors on Sharpe ratios at this horizon are ~0.3–0.5. (iii) No hyperparameter search beyond the paper-locked defaults; Lasso α, RF depth, and XGB lr were set to conservative values. (iv) Turnover was estimated on a 30-day sample for speed; production would require full-window computation.

**Implications for Session 3 (DL).** The linear/tree results give a clear baseline: any DL model (MLP, LSTM, Transformer) must materially beat the Sharpe of the best ML strategy here to justify its additional complexity and computational cost. The weak single-fit finding motivates an architecture that can incorporate time-varying regimes or rolling-refit schedules.

## §12 Save Artifacts

In [ ]:
# Build ML returns matrix (all test-period columns)
ml_returns = pd.DataFrame({
    'EW':                  ret_ew,
    'SignalTilt(mom_252)': ret_tilt_mom,
    'SignalTilt(Lasso)':   ret_lasso,
    'SignalTilt(RF)':      ret_rf,
    'SignalTilt(XGB)':     ret_xgb,
    'MSR(Lasso_μ̂)':       ret_msr_lasso,
    'MSR(RF_μ̂)':          ret_msr_rf,
    'MSR(XGB_μ̂)':         ret_msr_xgb,
}).loc[test_idx]

ml_returns.index.name = 'Date'
ml_returns.to_parquet(out_dir / 'ml_strategies_29assets.parquet')
print(f'ML returns: {ml_returns.shape} → {out_dir}/ml_strategies_29assets.parquet')
print(f'Comparison table: already saved')
print(f'Permutation importance figure: {ROOT}/docs/figures/rf_permutation_importance.png')

# Validation writeup
val_dir = ROOT / 'docs/validation'
val_dir.mkdir(parents=True, exist_ok=True)

top5_imp = rf_importance.sort_values(ascending=False).head(5)
imp_md = '\n'.join(f'| {k} | {v:.5f} |' for k, v in top5_imp.items())

table_md_lines = ['| Strategy | Ann Ret | Ann Vol | Sharpe | Max DD |', '|---|---|---|---|---|']
for strat_name, row in comparison.iterrows():
    table_md_lines.append(
        f"| {strat_name} | {row['Ann Ret']:.3f} | {row['Ann Vol']:.3f} | {row['Sharpe']:.3f} | {row['Max DD']:.3f} |"
    )
table_md = '\n'.join(table_md_lines)

val_md = f"""# Session 2 — ML Strategies Validation

## Methodology

Three ML strategies (Lasso, Random Forest, XGBoost) are trained once on the full training window (~2003–2019, after 1-year warmup) and evaluated out-of-sample on the test period (2023-01-01 to 2026-04-30, ~3.3 years). The target is the 21-day forward cumulative return. Features: 10 numeric (momentum, volatility, RSI, ATR ratio, Bollinger pct, overnight gap, volume z-score) + 7 asset-class one-hot dummies. The panel has 17 features × (date × 29 assets).

Two portfolio construction approaches: **Approach B** (core) wraps ML cross-sectional z-scores in a SignalTilt overlay (EW base + tilt_strength=0.5). **Approach A** (extension) feeds ML μ̂ directly into MSR (long-only clip + renormalize, Ledoit-Wolf Σ, lookback=504 days).

Train/validation/test split (paper-locked): `TRAIN_END=2022-12-31`, `TEST_START=2023-01-01`, validation = last 15% of pre-test window.

**Comparison note:** ML strategies are evaluated on true OOS data; paper baselines use full-sample Sharpes. This is an intentional apples-to-pears comparison by design.

## Headline Results (test period 2023–2026, sorted by Sharpe)

{table_md}

## Permutation Importance Findings (RF, validation set)

| Feature | Mean Importance |
|---|---|
{imp_md}

Momentum (252-day) and volatility features dominate. Asset-class dummies carry moderate importance, confirming within-class heterogeneity is informative. Short-horizon momentum (mom_21) ranks near the bottom, matching its near-zero IC from Session 1.5B.

## Limitations

1. **Single fit:** Model trained 2003–2019, never retrained. A rolling-refit strategy (Session 2c) would better adapt to regime shifts.
2. **Short test window:** 3.3 years yields Sharpe standard errors of ~0.3–0.5; cross-strategy differences may not be statistically significant.
3. **No hyperparameter tuning:** Conservative defaults throughout. Lasso α=1e-4, RF max_depth=8, XGB lr=0.05 set without grid search.
4. **MSR instability:** Approach A amplifies estimation error in μ̂ (Michaud 1989); the long-only clip is a heuristic fix, not a solution.

## Implications

**Session 3 (DL):** MLP/LSTM/Transformer architectures should aim to beat the best Approach B Sharpe here. The weak single-fit finding suggests time-varying architectures (LSTM with rolling context, Transformer attention) may be needed.

**Session 2c (optional rolling refit):** A walk-forward refit (e.g., annual refit on expanding window) would turn the single-fit experiment into a production-grade backtesting framework and likely close the gap vs. classical baselines.
"""

with open(val_dir / 'session_2_ml_strategies.md', 'w') as f:
    f.write(val_md)
print(f'Validation doc saved: {val_dir}/session_2_ml_strategies.md')

In [ ]:
# Final summary
print('=== Session 2b Artifacts ===')
print(f'ML returns parquet:       {(out_dir / "ml_strategies_29assets.parquet").exists()}')
print(f'Comparison table CSV:     {(out_dir / "ml_strategies_comparison_table.csv").exists()}')
print(f'RF importance figure:     {(ROOT / "docs/figures/rf_permutation_importance.png").exists()}')
print(f'Validation doc:           {(val_dir / "session_2_ml_strategies.md").exists()}')
print()
print(f'Fit times — Lasso: {lasso_time:.1f}s  RF: {rf_time:.1f}s  XGB: {xgb_time:.1f}s')
print()
print('Headline table (final):')
print(fmt.to_string())

## Extensions & Robustness Probes

The core ML story concludes at §11. Sections §13–§17 are optional deep-dives: VMP overlay, ensemble, apples-to-apples OOS comparison, extended comparison, hyperparameter sensitivity, and walk-forward refit.

## §13 VMP Overlay on ML Strategies

VMP (volatility-managed portfolio) overlay scales each strategy's exposure inversely to its recent realized volatility, targeting the strategy's own long-run vol. This is the same overlay applied to all 31 base strategies in the paper. Wraps each of the 6 ML strategies via `apply_vmp_overlay(returns, lookback=21, clip=(0.25, 1.5))` with `target_vol = ret.std() * np.sqrt(TRADING_DAYS)`.

In [ ]:
vmp_returns = {}
for name, ret in [
    ('VMP(SignalTilt(Lasso))', ret_lasso),
    ('VMP(SignalTilt(RF))',    ret_rf),
    ('VMP(SignalTilt(XGB))',   ret_xgb),
    ('VMP(MSR(Lasso_μ̂))',     ret_msr_lasso),
    ('VMP(MSR(RF_μ̂))',        ret_msr_rf),
    ('VMP(MSR(XGB_μ̂))',       ret_msr_xgb),
]:
    target = ret.dropna().std() * np.sqrt(TRADING_DAYS)
    vmp_returns[name] = apply_vmp_overlay(ret.rename(name), lookback=21, clip=(0.25, 1.5), target_vol=target)

vmp_metrics = []
for name, r in vmp_returns.items():
    r_test = r.reindex(test_idx).dropna()
    vmp_metrics.append({
        'Strategy': name,
        'Ann Ret': ann_return(r_test),
        'Ann Vol': ann_vol(r_test),
        'Sharpe':  ann_sharpe(r_test),
        'Max DD':  max_drawdown(r_test),
    })
vmp_df = pd.DataFrame(vmp_metrics).set_index('Strategy')
print('VMP-overlay ML strategies (test period):')
print(vmp_df.round(3).to_string())

## §14 Ensemble Averaged Predictions

Simple equal-weighted average of the three model predictions: `μ̂_ens = (μ̂_lasso + μ̂_rf + μ̂_xgb) / 3`. Wired through both portfolio construction approaches — SignalTilt wrapping and MSR with ML μ̂. No retraining; pure ensemble of existing model outputs.

In [ ]:
# Build ensemble predictions
ens_predictions = (lasso_strat.predictions + rf_strat.predictions + xgb_strat.predictions) / 3.0

class _EnsembleStrategy:
    def __init__(self, predictions, tilt_strength=0.5):
        self.predictions = predictions
        self.tilt_strength = tilt_strength
    def predict_weights(self, panel, asof):
        assets = panel.universe_at(asof)
        n = len(assets)
        base_w = pd.Series(1.0/n, index=assets)
        try:
            score = self.predictions.xs(asof, level=0)
        except KeyError:
            return base_w.rename(asof)
        score = score.reindex(assets).fillna(0.0)
        std = score.std()
        zs = (score - score.mean()) / std if std > 1e-12 else pd.Series(0.0, index=assets)
        w = (base_w + self.tilt_strength * zs).clip(lower=0.0)
        total = w.sum()
        return (w / total if total > 1e-12 else base_w).rename(asof)

ens_strat = _EnsembleStrategy(ens_predictions, tilt_strength=0.5)
ret_signaltilt_ens = walk_forward_returns(ens_strat, eval_dates, returns, panel)

def _msr_walk_forward_ensemble(predictions, eval_dates, returns_wide, cov_lookback=504):
    ret_series = {}
    for d in eval_dates[:-1]:
        try:
            mu_hat = predictions.xs(d, level=0)
        except KeyError:
            continue
        hist_window = returns_wide.loc[:d].iloc[-cov_lookback:].dropna(axis=1, how='all')
        common = hist_window.columns.intersection(mu_hat.index)
        if len(common) < 5:
            continue
        cov = ledoit_wolf_cov(hist_window[common].dropna())
        cov_inv = np.linalg.pinv(cov + 1e-8 * np.eye(len(common)))
        mu = mu_hat[common].values
        w_unc = cov_inv @ mu
        w = pd.Series(w_unc, index=common).clip(lower=0)
        total = w.sum()
        if total <= 1e-12:
            w = pd.Series(1.0/len(common), index=common)
        else:
            w = w / total
        next_d_mask = returns_wide.index > d
        if not next_d_mask.any():
            continue
        next_d = returns_wide.index[next_d_mask][0]
        r_next = returns_wide.loc[next_d]
        ret_series[next_d] = (w.reindex(r_next.index).fillna(0) * r_next).sum()
    return pd.Series(ret_series)

ret_msr_ens = _msr_walk_forward_ensemble(ens_predictions, eval_dates, returns)

ens_metrics = []
for name, r in [('SignalTilt(Ensemble)', ret_signaltilt_ens), ('MSR(Ensemble_μ̂)', ret_msr_ens)]:
    r_test = r.reindex(test_idx).dropna()
    ens_metrics.append({
        'Strategy': name,
        'Ann Ret': ann_return(r_test),
        'Ann Vol': ann_vol(r_test),
        'Sharpe':  ann_sharpe(r_test),
        'Max DD':  max_drawdown(r_test),
    })
ens_df = pd.DataFrame(ens_metrics).set_index('Strategy')
print('Ensemble strategies (test period):')
print(ens_df.round(3).to_string())

## §14b Apples-to-Apples: ML vs Deterministic OOS (2023–2026)

The headline table compared ML OOS Sharpes against classical *full-sample* Sharpes — methodologically valid (classical strategies are not penalised for in-sample fitting) but not the cleanest basis of comparison. The fairer question: how does the best ML strategy compare to the best deterministic strategy **on the same 2023–2026 window**?

On this apples-to-apples basis the ML edge is **modest but real**. MSR(Ensemble_μ̂) at ~2.58 beats the best deterministic OOS leader (VMP(MDP(LW)) at ~2.43) by roughly 0.15 Sharpe. That is meaningful outperformance — but it is a far cry from the 2.58-vs-1.37 gap the full-sample framing implies. The full-sample gap is largely an artefact of different comparison windows: deterministic strategies are measured on 20+ years including their best in-sample periods; ML strategies are measured on 3.3 years of true OOS. Paper 2's verdict turns on the 0.15 number, not the 1.2.

In [ ]:
# Apples-to-apples: deterministic strategies on the same 2023-2026 OOS window
_all_det = pd.concat([base_pub, vmp_pub], axis=1)
_oos_det = _all_det[_all_det.index >= TEST_START].copy()

# Exclude known artifact strategy (extreme leverage during 2008)
_artifact_cols = [c for c in _oos_det.columns if 'GMV(sample)' in c]
_oos_det = _oos_det.drop(columns=_artifact_cols, errors='ignore')

_det_sharpes = _oos_det.apply(ann_sharpe).sort_values(ascending=False)
_best_det_oos  = _det_sharpes.iloc[0]
_best_det_name = _det_sharpes.index[0]
print(f"Best deterministic OOS leader: {_best_det_name.replace('ledoit_wolf','LW')} = {_best_det_oos:.3f}")

# ML OOS Sharpes — reuse already-computed in-memory returns, no retraining
_ml_results = {
    'MSR(Ensemble_\u03bc\u0302)':  ann_sharpe(ret_msr_ens.reindex(test_idx).dropna()),
    'MSR(RF_\u03bc\u0302)':        ann_sharpe(ret_msr_rf.reindex(test_idx).dropna()),
    'SignalTilt(XGB)':              ann_sharpe(ret_xgb.reindex(test_idx).dropna()),
    'MSR(Lasso_\u03bc\u0302)':     ann_sharpe(ret_msr_lasso.reindex(test_idx).dropna()),
    'SignalTilt(RF)':               ann_sharpe(ret_rf.reindex(test_idx).dropna()),
}

_top_det = _det_sharpes.head(5)
_DET_COLOR = FAMILY_COLORS.get('Diversification', '#2ca02c')
_ML_COLOR  = '#c0392b'   # distinct red for ML family

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Apples-to-Apples: ML vs Deterministic (OOS 2023\u20132026)', fontsize=12, y=1.01)

# Left panel: ML strategies
_ml_labels = list(_ml_results.keys())
_ml_vals   = [_ml_results[k] for k in _ml_labels]
bars_ml = axes[0].barh(_ml_labels, _ml_vals, color=_ML_COLOR, alpha=0.85)
axes[0].axvline(_best_det_oos, color='#1f4e79', lw=1.8, ls='--',
                label=f'Best det. OOS {_best_det_oos:.2f}')
axes[0].set_title('ML Strategies (OOS Sharpe)')
axes[0].set_xlabel('Ann. Sharpe (2023\u20132026)')
axes[0].legend(fontsize=9)
for bar, val in zip(bars_ml, _ml_vals):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)

# Right panel: deterministic strategies
_det_disp = [n.replace('ledoit_wolf', 'LW') for n in _top_det.index]
_det_vals = list(_top_det.values)
bars_det = axes[1].barh(_det_disp, _det_vals, color=_DET_COLOR, alpha=0.85)
axes[1].axvline(_ml_results['MSR(Ensemble_\u03bc\u0302)'], color=_ML_COLOR, lw=1.8, ls='--',
                label=f'Best ML OOS {_ml_results["MSR(Ensemble_\u03bc\u0302)"]:.2f}')
axes[1].set_title('Deterministic Strategies (OOS Sharpe)')
axes[1].set_xlabel('Ann. Sharpe (2023\u20132026)')
axes[1].legend(fontsize=9)
for bar, val in zip(bars_det, _det_vals):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n=== OOS-vs-OOS Verdict (2023\u20132026) ===")
print(f"{'Strategy':<32}  {'Sharpe':>7}  Type")
print("-" * 58)
for strat, sh in sorted(_ml_results.items(), key=lambda x: -x[1]):
    delta = sh - _best_det_oos
    sign = f'+{delta:.3f}' if delta >= 0 else f'{delta:.3f}'
    print(f"{strat:<32}  {sh:7.3f}  ML  ({sign} vs best det.)")
print()
for name, sh in _top_det.items():
    print(f"{name.replace('ledoit_wolf','LW'):<32}  {sh:7.3f}  Deterministic")
_ml_edge = _ml_results['MSR(Ensemble_\u03bc\u0302)'] - _best_det_oos
print(f"\nHonest ML edge: {_ml_edge:+.3f} Sharpe (MSR(Ensemble_\u03bc\u0302) vs VMP(MDP(LW)))")
print(f"Full-sample framing gap (misleading): {_ml_results['MSR(Ensemble_\u03bc\u0302)'] - 1.37:.3f}")

## §15 Extended Comparison Table (19 strategies)

All 11 original strategies plus 6 VMP-wrapped ML variants plus 2 ensemble variants = 19 strategies sorted by Sharpe. The full picture of what ML brings to this cross-asset universe under different portfolio construction overlays.

In [ ]:
extended_rows = []
ml_base_names = {'SignalTilt(Lasso)','SignalTilt(RF)','SignalTilt(XGB)','MSR(Lasso_μ̂)','MSR(RF_μ̂)','MSR(XGB_μ̂)'}
classical_names = {'EW','MSR(LW)','SWITCH(v2a)','VMP(MDP(LW))'}
for strat_name, row in comparison.iterrows():
    if strat_name in classical_names:
        family = 'Classical'
    elif strat_name in ml_base_names:
        family = 'ML (single-fit)'
    else:
        family = 'Classical'
    extended_rows.append({
        'Strategy': strat_name, 'Ann Ret': row['Ann Ret'], 'Ann Vol': row['Ann Vol'],
        'Sharpe': row['Sharpe'], 'Max DD': row['Max DD'], 'Family': family,
    })
for name, r in vmp_returns.items():
    r_test = r.reindex(test_idx).dropna()
    extended_rows.append({
        'Strategy': name, 'Ann Ret': ann_return(r_test), 'Ann Vol': ann_vol(r_test),
        'Sharpe': ann_sharpe(r_test), 'Max DD': max_drawdown(r_test), 'Family': 'ML + VMP',
    })
for name, r in [('SignalTilt(Ensemble)', ret_signaltilt_ens), ('MSR(Ensemble_μ̂)', ret_msr_ens)]:
    r_test = r.reindex(test_idx).dropna()
    extended_rows.append({
        'Strategy': name, 'Ann Ret': ann_return(r_test), 'Ann Vol': ann_vol(r_test),
        'Sharpe': ann_sharpe(r_test), 'Max DD': max_drawdown(r_test), 'Family': 'ML (ensemble)',
    })

extended = pd.DataFrame(extended_rows).set_index('Strategy').sort_values('Sharpe', ascending=False)
print(f'Extended comparison: {len(extended)} strategies')
print(extended[['Family','Ann Ret','Ann Vol','Sharpe','Max DD']].round(3).to_string())

extended.to_csv(out_dir / 'ml_strategies_extended_comparison.csv')

ml_returns_extended = ml_returns.copy()
for name, r in vmp_returns.items():
    ml_returns_extended[name] = r.reindex(test_idx)
ml_returns_extended['SignalTilt(Ensemble)'] = ret_signaltilt_ens.reindex(test_idx)
ml_returns_extended['MSR(Ensemble_μ̂)'] = ret_msr_ens.reindex(test_idx)
ml_returns_extended.to_parquet(out_dir / 'ml_strategies_29assets_extended.parquet')
print(f'\nSaved: {out_dir}/ml_strategies_extended_comparison.csv')
print(f'Saved: {out_dir}/ml_strategies_29assets_extended.parquet ({ml_returns_extended.shape})')

## §16 Hyperparameter Sensitivity Diagnostic

The defaults used in §4–§6 come from authoritative sources: JPM ML Quant Equity Risk Factors (Feb 2025) for the RF specs (100 trees, max_depth=8, min_samples_leaf=50), Hilpisch §14 for tree-based defaults, and standard daily-return regime for Lasso (α=1e-4). This section confirms whether these defaults are near-optimal on the validation window (~2019-10 → 2022-12-30) or whether a small grid search reveals materially better hyperparameters. Validation metrics: rank IC (cross-sectional Spearman correlation between predicted and forward returns).


In [ ]:
from sklearn.linear_model import Lasso
from aiam.ml.workflow import apply_standardizer
from aiam.evaluation.ic import information_coefficient

val_mask_long = X_full.index.get_level_values('Date').isin(val_dates)
X_val_long = X_full.loc[val_mask_long, FEATURE_COLS]
y_val_long = y_full.loc[val_mask_long]

lasso_grid = []
for alpha in [1e-5, 1e-4, 1e-3, 1e-2]:
    m = Lasso(alpha=alpha, random_state=42, max_iter=10_000)
    m.fit(lasso_strat._X_train, lasso_strat._y_train)
    val_arr = apply_standardizer(X_val_long, lasso_strat._center, lasso_strat._scale, FEATURE_COLS).values
    pred = pd.Series(m.predict(val_arr), index=X_val_long.index, name='pred')
    ic = information_coefficient(pred.unstack('Asset'), y_val_long.unstack('Asset'), method='spearman').mean()
    lasso_grid.append({'alpha': alpha, 'val_IC': round(ic, 5), 'n_nonzero': int((m.coef_ != 0).sum())})

lasso_df = pd.DataFrame(lasso_grid)
print('Lasso HP grid (validation set):')
print(lasso_df.to_string(index=False))
best = lasso_df.loc[lasso_df['val_IC'].idxmax()]
default_ic = lasso_df.loc[lasso_df['alpha'] == 1e-4, 'val_IC'].iloc[0]
print(f'\nDefault (α=1e-4): val_IC = {default_ic:.5f}')
print(f'Best alpha: {best["alpha"]:.0e}, val_IC = {best["val_IC"]:.5f}, Δ = {best["val_IC"] - default_ic:+.5f}')


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_grid = []
for (n_est, depth) in [(50, 6), (100, 8), (200, 10)]:
    m = RandomForestRegressor(n_estimators=n_est, max_depth=depth, min_samples_leaf=50,
                              n_jobs=-1, random_state=42)
    m.fit(rf_strat._X_train, rf_strat._y_train)
    val_arr = apply_standardizer(X_val_long, rf_strat._center, rf_strat._scale, FEATURE_COLS).values
    pred = pd.Series(m.predict(val_arr), index=X_val_long.index)
    ic = information_coefficient(pred.unstack('Asset'), y_val_long.unstack('Asset'), method='spearman').mean()
    rf_grid.append({'n_estimators': n_est, 'max_depth': depth, 'val_IC': round(ic, 5)})

rf_df = pd.DataFrame(rf_grid)
print('Random Forest HP grid (validation set):')
print(rf_df.to_string(index=False))
best = rf_df.loc[rf_df['val_IC'].idxmax()]
default_ic = rf_df.loc[(rf_df['n_estimators'] == 100) & (rf_df['max_depth'] == 8), 'val_IC'].iloc[0]
print(f'\nDefault (100 trees, depth=8): val_IC = {default_ic:.5f}')
print(f'Best: ({int(best["n_estimators"])}, depth={int(best["max_depth"])}), val_IC = {best["val_IC"]:.5f}, Δ = {best["val_IC"] - default_ic:+.5f}')


In [ ]:
import xgboost as xgb

xgb_grid = []
for (n_est, lr, depth) in [(200, 0.05, 4), (300, 0.05, 6), (500, 0.03, 6), (300, 0.10, 4)]:
    m = xgb.XGBRegressor(n_estimators=n_est, learning_rate=lr, max_depth=depth,
                         early_stopping_rounds=20, random_state=42, n_jobs=-1, tree_method='hist')
    m.fit(xgb_strat._X_train, xgb_strat._y_train,
          eval_set=[(xgb_strat._X_val, xgb_strat._y_val)], verbose=False)
    val_arr = apply_standardizer(X_val_long, xgb_strat._center, xgb_strat._scale, FEATURE_COLS).values
    pred = pd.Series(m.predict(val_arr), index=X_val_long.index)
    ic = information_coefficient(pred.unstack('Asset'), y_val_long.unstack('Asset'), method='spearman').mean()
    xgb_grid.append({'n_est': n_est, 'lr': lr, 'depth': depth,
                     'best_iter': m.best_iteration, 'val_IC': round(ic, 5)})

xgb_df = pd.DataFrame(xgb_grid)
print('XGBoost HP grid (validation set):')
print(xgb_df.to_string(index=False))
best = xgb_df.loc[xgb_df['val_IC'].idxmax()]
default_ic = xgb_df.loc[(xgb_df['n_est']==300)&(xgb_df['lr']==0.05)&(xgb_df['depth']==6), 'val_IC'].iloc[0]
print(f'\nDefault (300, lr=0.05, depth=6): val_IC = {default_ic:.5f}')
print(f'Best: ({int(best["n_est"])}, lr={best["lr"]}, depth={int(best["depth"])}), val_IC = {best["val_IC"]:.5f}, Δ = {best["val_IC"] - default_ic:+.5f}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Lasso
labels_l = [f'α={a:.0e}' for a in lasso_df['alpha']]
colors_l = [ML_FAMILY_COLORS['ML single-fit'] if a != 1e-4 else '#2ca02c' for a in lasso_df['alpha']]
axes[0].bar(labels_l, lasso_df['val_IC'], color=colors_l, alpha=0.85, edgecolor='white')
axes[0].set_title('Lasso — α sensitivity', fontsize=10)
axes[0].set_ylabel('Validation rank IC', fontsize=9)
axes[0].set_xlabel('alpha (green = default)', fontsize=8)
axes[0].spines[['top', 'right']].set_visible(False)

# RF
labels_r = [f'({n},d{d})' for n, d in zip(rf_df['n_estimators'], rf_df['max_depth'])]
colors_r = ['#2ca02c' if (n == 100 and d == 8) else ML_FAMILY_COLORS['ML single-fit']
            for n, d in zip(rf_df['n_estimators'], rf_df['max_depth'])]
axes[1].bar(labels_r, rf_df['val_IC'], color=colors_r, alpha=0.85, edgecolor='white')
axes[1].set_title('RF — (n_est, depth) sensitivity', fontsize=10)
axes[1].set_xlabel('(n_est, depth) (green = default)', fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)

# XGB
labels_x = [f'({r["n_est"]},{r["lr"]},{r["depth"]})' for _, r in xgb_df.iterrows()]
colors_x = ['#2ca02c' if (r['n_est']==300 and r['lr']==0.05 and r['depth']==6)
            else ML_FAMILY_COLORS['ML single-fit'] for _, r in xgb_df.iterrows()]
axes[2].bar(labels_x, xgb_df['val_IC'], color=colors_x, alpha=0.85, edgecolor='white')
axes[2].set_title('XGB — (n_est, lr, depth) sensitivity', fontsize=10)
axes[2].set_xlabel('(n_est, lr, depth) (green = default)', fontsize=8)
axes[2].tick_params(axis='x', labelsize=7)
axes[2].spines[['top', 'right']].set_visible(False)

plt.suptitle('Validation-set Rank IC by Hyperparameter Configuration', fontsize=11)
plt.tight_layout()
fig_dir = ROOT / 'docs/figures/session2'
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / 'hp_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session2/hp_sensitivity.png')


**HP Sensitivity Findings**

| Model | Default val_IC | Best val_IC | Δ | Best config |
|---|---|---|---|---|
| Lasso | 0.112 | 0.131 | +0.019 | α=1e-3 (4 non-zero coefs) |
| RF | 0.038 | 0.056 | +0.018 | 50 trees, depth=6 |
| XGBoost | 0.013 | 0.086 | +0.073 | 200 trees, lr=0.05, depth=4 |

**Lasso.** Default (α=1e-4) is near-optimal but α=1e-3 is marginally better (+17% relative). The sparser model (4 vs 15 non-zero features) shows the default is slightly under-regularized. α=1e-2 collapses all coefficients to zero (NaN IC).

**RF.** Default (100 trees, depth=8) is outperformed by the shallower (50 trees, depth=6) by +48% relative. Deeper trees overfit on this 29-asset cross-asset universe. The JPM large-universe spec (max_depth=8) is too complex here; the low asset count limits the cross-sectional diversity that deep trees exploit.

**XGBoost.** Default (300, lr=0.05, depth=6) shows `best_iteration=0` — early stopping triggers after the very first boosting round, indicating catastrophic overfitting at depth=6 on this dataset. Reducing to depth=4 rescues signal (val_IC 0.013 → 0.086, +583% relative). Notably, the depth=6 default ranked #3 OOS (Sharpe 2.304); the val/OOS disagreement suggests regime shift between 2019–2022 (val) and 2023–2026 (OOS).

**Decision.** Keep current OOS comparison unchanged — the 19-strategy table is the paper deliverable and replacing models mid-session would invalidate those results. Flag for Session 2c-B walk-forward: use α=1e-3 (Lasso), (50, depth=6) RF, (200, lr=0.05, depth=4) XGB. The XGB val/OOS disagreement is the sharpest finding: a model with catastrophic validation overfitting still ranks near the top OOS — worth probing in 2c-B with rolling refit.

In [ ]:
comparison_returns = {
    'EW': ret_ew,
    'SignalTilt(mom_252)': ret_tilt_mom,
    'SignalTilt(Lasso)': ret_lasso,
    'SignalTilt(RF)': ret_rf,
    'SignalTilt(XGB)': ret_xgb,
    'MSR(Lasso_μ̂)': ret_msr_lasso,
    'MSR(RF_μ̂)': ret_msr_rf,
    'MSR(XGB_μ̂)': ret_msr_xgb,
    'MSR(LW)': msr_lw_test,
    'SWITCH(v2a)': sw_test,
    'VMP(MDP(LW))': vmp_mdp_test,
    **vmp_returns,
    'SignalTilt(Ensemble)': ret_signaltilt_ens,
    'MSR(Ensemble_μ̂)': ret_msr_ens,
}
fig_dir = ROOT / 'docs/figures/session2'
fig_dir.mkdir(parents=True, exist_ok=True)
print(f'comparison_returns: {len(comparison_returns)} strategies')
print(f'Session 2 figures dir: {fig_dir}')


In [ ]:
top6 = extended.head(6).index.tolist()
ls_cycle = {fam: iter([('-', 2.0), ('--', 1.6), (':', 1.4)]) for fam in ML_FAMILY_COLORS}

fig, ax = plt.subplots(figsize=(11, 5))
for strat in top6:
    r = comparison_returns.get(strat, pd.Series(dtype=float)).dropna()
    if len(r) == 0:
        continue
    wealth = (1 + r).cumprod()
    fam = STRATEGY_FAMILY.get(strat, 'Classical')
    color = ML_FAMILY_COLORS[fam]
    try:
        ls, lw = next(ls_cycle[fam])
    except StopIteration:
        ls, lw = '-', 1.2
    ax.plot(wealth.index, wealth.values, label=strat, color=color, ls=ls, lw=lw)
ax.set_yscale('log')
ax.set_ylabel('Cumulative wealth (log, $1→)', fontsize=10)
ax.set_title('Cumulative wealth, top 6 strategies — test period 2023–2026', fontsize=11)
ax.legend(loc='upper left', fontsize=8, framealpha=0.85)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(fig_dir / 'cum_wealth_top6.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session2/cum_wealth_top6.png')


In [ ]:
top5 = extended.head(5).index.tolist()

fig, ax = plt.subplots(figsize=(11, 4))
for strat in top5:
    r = comparison_returns.get(strat, pd.Series(dtype=float)).dropna()
    if len(r) == 0:
        continue
    wealth = (1 + r).cumprod()
    dd = wealth / wealth.cummax() - 1
    fam = STRATEGY_FAMILY.get(strat, 'Classical')
    color = ML_FAMILY_COLORS[fam]
    ax.fill_between(dd.index, dd.values, 0, alpha=0.20, color=color)
    ax.plot(dd.index, dd.values, label=strat, color=color, lw=1.3)
ax.set_ylabel('Drawdown', fontsize=10)
ax.set_title('Underwater drawdown, top 5 strategies — test period 2023–2026', fontsize=11)
ax.legend(loc='lower left', fontsize=8, framealpha=0.85)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(fig_dir / 'drawdown_top5.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session2/drawdown_top5.png')


In [ ]:
top5 = extended.head(5).index.tolist()

fig, ax = plt.subplots(figsize=(11, 4))
for strat in top5:
    r = comparison_returns.get(strat, pd.Series(dtype=float)).dropna()
    if len(r) == 0:
        continue
    roll_sh = r.rolling(126).mean() / r.rolling(126).std() * np.sqrt(TRADING_DAYS)
    fam = STRATEGY_FAMILY.get(strat, 'Classical')
    ax.plot(roll_sh.index, roll_sh.values, label=strat, color=ML_FAMILY_COLORS[fam], lw=1.3)
ax.axhline(0, color='black', lw=0.7, ls='--')
ax.axhline(1, color='gray', lw=0.5, ls=':')
ax.set_ylabel('Rolling 126-day Sharpe', fontsize=10)
ax.set_title('Rolling 126-day Sharpe, top 5 strategies — test period 2023–2026', fontsize=11)
ax.legend(loc='upper left', fontsize=8, framealpha=0.85)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(fig_dir / 'rolling_sharpe_top5.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session2/rolling_sharpe_top5.png')


In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(9, 6))
for strat, row in extended.iterrows():
    fam = STRATEGY_FAMILY.get(strat, 'Classical')
    color = ML_FAMILY_COLORS[fam]
    marker = '*' if strat == 'MSR(Ensemble_μ̂)' else 'o'
    size = 150 if strat == 'MSR(Ensemble_μ̂)' else 55
    ax.scatter(row['Ann Vol'], row['Sharpe'], color=color, marker=marker, s=size, zorder=3, alpha=0.85)
    ax.annotate(strat, (row['Ann Vol'], row['Sharpe']),
                fontsize=5.5, xytext=(3, 3), textcoords='offset points', ha='left')
handles = [mpatches.Patch(color=c, label=f) for f, c in ML_FAMILY_COLORS.items()]
ax.legend(handles=handles, fontsize=8, loc='upper right')
ax.set_xlabel('Annualized Volatility', fontsize=10)
ax.set_ylabel('Sharpe Ratio', fontsize=10)
ax.set_title('Sharpe vs Volatility — 19 strategies (test period 2023–2026)', fontsize=11)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(fig_dir / 'sharpe_vs_vol_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session2/sharpe_vs_vol_scatter.png')


In [ ]:
years = [2023, 2024, 2025, 2026]
cal_sharpe = {}
for strat, r in comparison_returns.items():
    r = r.dropna()
    cal_sharpe[strat] = {}
    for yr in years:
        r_yr = r[r.index.year == yr]
        if len(r_yr) > 20:
            cal_sharpe[strat][yr] = r_yr.mean() / r_yr.std() * np.sqrt(TRADING_DAYS)
        else:
            cal_sharpe[strat][yr] = np.nan

cal_df = pd.DataFrame(cal_sharpe).T.reindex(extended.index)[years]
n_strats = len(cal_df)

fig, ax = plt.subplots(figsize=(6, max(7, n_strats * 0.45)))
vmin, vmax = -2, 4
im = ax.pcolormesh(cal_df.values, cmap=plt.cm.RdYlGn, vmin=vmin, vmax=vmax)
ax.set_xticks(np.arange(len(years)) + 0.5)
ax.set_xticklabels([str(y) for y in years], fontsize=10)
ax.set_yticks(np.arange(n_strats) + 0.5)
ax.set_yticklabels(cal_df.index, fontsize=8)
for i in range(n_strats):
    for j in range(len(years)):
        val = cal_df.values[i, j]
        if not np.isnan(val):
            text_color = 'black' if -1.5 < val < 3.5 else 'white'
            ax.text(j + 0.5, i + 0.5, f'{val:.2f}', ha='center', va='center',
                    fontsize=7, color=text_color)
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.03, label='Sharpe')
ax.set_title('Calendar-year Sharpe (sorted by overall Sharpe)', fontsize=11)
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(fig_dir / 'calendar_year_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session2/calendar_year_sharpe.png')


## §17 Walk-Forward Refit — Annual Rolling Window

Single-fit models (§4–§6) train once on 2003–2020 and predict through 2026 — a 3.3-year gap risks model decay across regime shifts. Walk-forward refit recomputes each model every January on the trailing 10 years of data, then predicts for the next 12 months. Two HP regimes compared: **(a) WF-default** uses the Hilpisch/JPM defaults from §4–§6; **(b) WF-val-optimal** uses the validation-tuned HPs from §16. If validation-tuning is biased by the COVID/rate-shock regime (as the §16 val_IC vs OOS Sharpe disagreement suggests), WF-default should win this comparison.

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb


def walk_forward_predictions(
    feature_df, target_series, feature_cols, numeric_cols,
    model_factory, refit_dates, lookback_years=10, train_end_buffer=21,
):
    """Retrain on trailing lookback_years at each refit date; predict for next 12 months.
    
    train_end_buffer (days) excludes the final forward-return horizon from training so the
    target labels do not overlap with the prediction window.
    """
    all_predictions = []
    feat_dates = feature_df.index.get_level_values("Date")
    tgt_dates  = target_series.index.get_level_values("Date")
    for refit_d in refit_dates:
        train_end   = refit_d - pd.Timedelta(days=train_end_buffer)
        train_start = refit_d - pd.DateOffset(years=lookback_years)
        X_tr = feature_df.loc[
            (feat_dates >= train_start) & (feat_dates <= train_end), feature_cols
        ]
        y_tr = target_series.loc[
            (tgt_dates >= train_start) & (tgt_dates <= train_end)
        ].dropna()
        valid_idx = X_tr.index.intersection(y_tr.index)
        X_tr, y_tr = X_tr.loc[valid_idx], y_tr.loc[valid_idx]
        center = X_tr[numeric_cols].mean()
        scale  = X_tr[numeric_cols].std().replace(0.0, 1.0)
        X_tr_s = X_tr.copy()
        X_tr_s[numeric_cols] = (X_tr[numeric_cols] - center) / scale
        model = model_factory()
        model.fit(X_tr_s.values, y_tr.values)
        pred_start = refit_d
        pred_end   = refit_d + pd.DateOffset(years=1) - pd.Timedelta(days=1)
        X_pred = feature_df.loc[
            (feat_dates >= pred_start) & (feat_dates <= pred_end), feature_cols
        ]
        X_pred_s = X_pred.copy()
        X_pred_s[numeric_cols] = (X_pred[numeric_cols] - center) / scale
        preds = pd.Series(model.predict(X_pred_s.values), index=X_pred.index, name="pred")
        all_predictions.append(preds)
    return pd.concat(all_predictions).sort_index()


# Refit on Jan 2 each year of the test period (2023-2026)
refit_dates = pd.to_datetime(["2023-01-02", "2024-01-02", "2025-01-02", "2026-01-02"])
print(f"Refit dates: {[d.date() for d in refit_dates]}")
print(f"Training window: trailing 10 years per refit date")
print(f"Prediction span: {refit_dates[0].date()} to end of 2026")

In [ ]:
WF_HP_DEFAULT = {
    'lasso': lambda: Lasso(alpha=1e-4, random_state=42, max_iter=10_000),
    'rf':    lambda: RandomForestRegressor(
        n_estimators=100, max_depth=8, min_samples_leaf=50, n_jobs=-1, random_state=42),
    'xgb':   lambda: xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42,
        n_jobs=-1, tree_method='hist'),
}
WF_HP_VAL_OPTIMAL = {
    'lasso': lambda: Lasso(alpha=1e-3, random_state=42, max_iter=10_000),
    'rf':    lambda: RandomForestRegressor(
        n_estimators=50, max_depth=6, min_samples_leaf=50, n_jobs=-1, random_state=42),
    'xgb':   lambda: xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42,
        n_jobs=-1, tree_method='hist'),
}

wf_preds = {}
for regime_name, hps in [('default', WF_HP_DEFAULT), ('val_optimal', WF_HP_VAL_OPTIMAL)]:
    for model_name, factory in hps.items():
        key = f'WF-{model_name}-{regime_name}'
        t0 = time.time()
        wf_preds[key] = walk_forward_predictions(
            feature_panel, y_full, FEATURE_COLS, NUMERIC_COLS,
            factory, refit_dates, lookback_years=10,
        )
        elapsed = time.time() - t0
        print(f'{key:35s}  fit: {elapsed:.1f}s  preds: {len(wf_preds[key]):,}')

In [ ]:
class _WFStrat:
    """Anonymous walk-forward strategy: SignalTilt using WF predictions."""
    def __init__(self, preds):
        self.predictions = preds

    def predict_weights(self, panel_obj, asof):
        assets = panel_obj.universe_at(asof)
        n = len(assets)
        base_w = pd.Series(1.0 / n, index=assets)
        try:
            score = self.predictions.xs(asof, level=0)
        except KeyError:
            return base_w.rename(asof)
        score = score.reindex(assets).fillna(0.0)
        std = score.std()
        zs = (score - score.mean()) / std if std > 1e-12 else pd.Series(0.0, index=assets)
        w = (base_w + 0.5 * zs).clip(lower=0.0)
        total = w.sum()
        return (w / total if total > 1e-12 else base_w).rename(asof)


wf_returns_st = {}
for key, preds in wf_preds.items():
    strat = _WFStrat(preds)
    r = walk_forward_returns(strat, eval_dates, returns, panel)
    wf_returns_st[f'SignalTilt({key})'] = r

wf_st_metrics = []
for name, r in wf_returns_st.items():
    r_test = r.reindex(test_idx).dropna()
    wf_st_metrics.append({
        'Strategy': name,
        'Ann Ret': ann_return(r_test), 'Ann Vol': ann_vol(r_test),
        'Sharpe':  ann_sharpe(r_test), 'Max DD':  max_drawdown(r_test),
    })
wf_st_df = pd.DataFrame(wf_st_metrics).set_index('Strategy').sort_values('Sharpe', ascending=False)
print('Walk-forward SignalTilt strategies (test period):')
print(wf_st_df.round(3).to_string())

In [ ]:
# MSR(WF-*-default) only — val_optimal doubles the table for marginal gain
wf_returns_msr = {}
for model_name in ['lasso', 'rf', 'xgb']:
    key = f'WF-{model_name}-default'
    preds = wf_preds[key]
    r = _msr_walk_forward_ensemble(preds, eval_dates, returns)
    wf_returns_msr[f'MSR(WF-{model_name}-default_\u03bc\u0302)'] = r

wf_msr_metrics = []
for name, r in wf_returns_msr.items():
    r_test = r.reindex(test_idx).dropna()
    wf_msr_metrics.append({
        'Strategy': name,
        'Ann Ret': ann_return(r_test), 'Ann Vol': ann_vol(r_test),
        'Sharpe':  ann_sharpe(r_test), 'Max DD':  max_drawdown(r_test),
    })
print('Walk-forward MSR strategies (test period, default HPs):')
print(pd.DataFrame(wf_msr_metrics).set_index('Strategy').round(3).to_string())

In [ ]:
# Map single-fit returns by model name for comparison
sf_ret_map = {'rf': ret_rf, 'xgb': ret_xgb}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for model_name, ax in zip(['rf', 'xgb'], axes):
    sf_ret   = sf_ret_map[model_name]
    wf_ret   = wf_returns_st[f'SignalTilt(WF-{model_name}-default)']
    wf_val_ret = wf_returns_st[f'SignalTilt(WF-{model_name}-val_optimal)']
    for r, label, color, ls in [
        (sf_ret,     f'Single-fit {model_name.upper()}',    '#d62728', '-'),
        (wf_ret,     f'WF-{model_name}-default',            '#1f4e79', '--'),
        (wf_val_ret, f'WF-{model_name}-val-optimal',        '#2ca02c', ':'),
    ]:
        wealth = (1 + r.reindex(test_idx).dropna()).cumprod()
        ax.plot(wealth.index, wealth.values, label=label, color=color, linestyle=ls, linewidth=1.5)
    ax.set_yscale('log')
    ax.set_title(f'Walk-forward vs single-fit — {model_name.upper()}', fontsize=11)
    ax.set_ylabel('Cumulative wealth (log scale)')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig_dir = ROOT / 'docs/figures/session2'
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / 'wf_vs_single_fit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved wf_vs_single_fit.png')

In [ ]:
# Extended v2: 19 → 28 strategies
extended_v2 = extended.copy()

for name, r in wf_returns_st.items():
    r_test = r.reindex(test_idx).dropna()
    hp = 'Walk-forward (val-optimal HPs)' if 'val_optimal' in name else 'Walk-forward (default HPs)'
    extended_v2.loc[name] = {
        'Family': hp, 'Ann Ret': ann_return(r_test),
        'Ann Vol': ann_vol(r_test), 'Sharpe': ann_sharpe(r_test),
        'Max DD': max_drawdown(r_test),
    }
for name, r in wf_returns_msr.items():
    r_test = r.reindex(test_idx).dropna()
    extended_v2.loc[name] = {
        'Family': 'Walk-forward (default HPs)', 'Ann Ret': ann_return(r_test),
        'Ann Vol': ann_vol(r_test), 'Sharpe': ann_sharpe(r_test),
        'Max DD': max_drawdown(r_test),
    }

extended_v2 = extended_v2.sort_values('Sharpe', ascending=False)
print(f'Extended v2 comparison: {len(extended_v2)} strategies')
print(extended_v2[['Family', 'Ann Ret', 'Ann Vol', 'Sharpe', 'Max DD']].round(3).to_string())

extended_v2.to_csv(out_dir / 'ml_strategies_extended_v2_comparison.csv')

# Save walk-forward returns parquet
all_wf_returns = pd.DataFrame({**wf_returns_st, **wf_returns_msr}).reindex(test_idx)
all_wf_returns.to_parquet(out_dir / 'ml_strategies_walk_forward.parquet')
print(f'\nSaved extended_v2_comparison.csv and walk_forward.parquet to {out_dir}')

**Walk-Forward Findings (Session 2c-B)**

| Strategy | Ann Ret | Ann Vol | Sharpe | Max DD |
|---|---|---|---|---|
| SignalTilt(WF-lasso-default)    | 0.552 | 0.229 | 2.033 | -0.231 |
| SignalTilt(WF-lasso-val-opt)    | 0.566 | 0.238 | 2.006 | -0.229 |
| MSR(WF-lasso-default_μ̂)        | 0.246 | 0.114 | 1.987 | -0.141 |
| MSR(WF-xgb-default_μ̂)          | 0.238 | 0.111 | 1.984 | -0.097 |
| SignalTilt(WF-xgb-val-opt)      | 0.454 | 0.214 | 1.860 | -0.264 |
| MSR(WF-rf-default_μ̂)           | 0.248 | 0.120 | 1.913 | -0.104 |
| SignalTilt(WF-xgb-default)      | 0.370 | 0.187 | 1.777 | -0.169 |
| SignalTilt(WF-rf-val-opt)       | 0.470 | 0.236 | 1.751 | -0.250 |
| SignalTilt(WF-rf-default)       | 0.394 | 0.221 | 1.617 | -0.229 |

**Single-fit vs walk-forward Sharpe comparison:**

| Model | Single-fit | WF-default | WF-val-optimal |
|---|---|---|---|
| Lasso | 2.140 | **2.033** | 2.006 |
| RF    | 2.252 | 1.617 | **1.751** |
| XGB   | 2.304 | 1.777 | **1.860** |

- **Did walk-forward beat single-fit?** No. All 9 WF strategies ranked below their single-fit counterparts. Single-fit models trained 2003–2020 (including the COVID stress + early rate shock regime) appear to encode regime-appropriate feature weighting better than WF models trained only on the trailing 10 years. The 3.3-year single-fit gap did not cause detectable model decay on this test period.

- **Did default HPs beat val-optimal in walk-forward?** Mixed. Lasso: default wins (2.033 vs 2.006) — consistent with the validation-bias hypothesis. RF and XGB: val-optimal wins (+0.134 and +0.083 Sharpe respectively) — the §16 tuning generalises in walk-forward for tree-based models. Net verdict: validation bias is model-specific; Lasso is the only case where it clearly operated.

- **Did MSR(WF-*) outperform MSR(single-fit)?** No. MSR(WF-lasso): 1.987 vs 2.272; MSR(WF-rf): 1.913 vs 2.394; MSR(WF-xgb): 1.984 vs 2.180. Walk-forward refit degraded MSR performance more than SignalTilt, consistent with MSR amplifying estimation error from the noisier walk-forward predictions.

- **Surprises:** The single-fit regime being more informative than rolling refits suggests the 2020–2022 market regime (COVID, rate hike cycle) is predictive of 2023–2026 returns — i.e., the test period is still a post-rate-shock environment where the single-fit model's memory of that regime is an asset, not a liability.